<a href="https://colab.research.google.com/github/dtoralg/INESDI_Data-Science_ML_IA/blob/main/%5B08%5D%20-%20NLP/nlp_tweets_publico_completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP aplicado a negocio con un dataset público de X/Twitter  
## Análisis de sentimiento sobre tweets de aerolíneas

Este notebook sirve como **introducción completa y práctica al flujo de trabajo típico en NLP clásico** usando un dataset **real y público** de mensajes publicados en Twitter/X sobre aerolíneas de EE. UU.

La idea no es solo entrenar un modelo, sino recorrer el flujo completo:

1. Cargar un dataset real.
2. Entender qué problema de negocio estamos resolviendo.
3. Explorar los datos.
4. Limpiar el texto.
5. Convertir texto en variables numéricas.
6. Entrenar varios modelos de clasificación.
7. Evaluar resultados.
8. Interpretar qué ha aprendido el modelo.
9. Conectar el ejercicio con casos reales de negocio.

> **Caso de negocio**: una empresa quiere monitorizar automáticamente mensajes de clientes en redes sociales para detectar si el sentimiento es positivo, neutral o negativo, y así priorizar incidencias, medir reputación y alimentar dashboards o flujos de atención al cliente.

## 1. Objetivos del notebook

Al finalizar este notebook deberías tener clara una visión bastante completa de:

- cómo se estructura un problema de NLP en clasificación de texto,
- por qué el texto no puede entrar "tal cual" a un modelo clásico,
- cómo funciona una representación **TF-IDF**,
- cómo evaluar un clasificador de texto,
- cómo interpretar resultados y errores,
- y dónde encaja esto dentro de un caso de negocio real.


## 2. Dataset público que vamos a utilizar

Usaremos el dataset **Twitter US Airline Sentiment**, muy conocido en NLP introductorio.

Contiene tweets reales dirigidos a aerolíneas como:

- United
- American
- Delta
- Southwest
- US Airways
- Virgin America

Y para cada tweet tenemos una etiqueta de sentimiento:

- `positive`
- `neutral`
- `negative`

Esto lo convierte en un ejemplo muy bueno de negocio para:

- social listening,
- customer support,
- reputación de marca,
- análisis de voz del cliente,
- detección temprana de incidencias.

En este notebook cargaremos una copia pública accesible vía URL.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

## 3. Carga de datos

Vamos a descargar el dataset desde una fuente pública.

Si esta URL fallase en algún momento, la alternativa en clase sería descargar el CSV previamente y leerlo desde una ruta local.

In [2]:
DATA_URL = "https://raw.githubusercontent.com/kolaveridi/kaggle-Twitter-US-Airline-Sentiment-/master/Tweets.csv"

df = pd.read_csv(DATA_URL)
print("Shape:", df.shape)
df.head()

HTTPError: HTTP Error 404: Not Found

## 4. Primera inspección del dataset

Antes de modelar, toca mirar qué tenemos entre manos.  
La prisa en ML suele salir cara. En NLP, más todavía.

In [ ]:
df.info()

In [ ]:
df.columns.tolist()

Nos vamos a quedar con un subconjunto útil de variables para este ejercicio:

- `text`: contenido del tweet
- `airline_sentiment`: etiqueta objetivo
- `airline`: aerolínea mencionada
- `negativereason`: motivo del sentimiento negativo cuando aplica
- `tweet_created`: fecha/hora del tweet

In [ ]:
cols = ["text", "airline_sentiment", "airline", "negativereason", "tweet_created"]
df = df[cols].copy()

print(df.shape)
df.head()

## 5. Revisión básica de calidad de datos

In [ ]:
df.isna().sum()

In [ ]:
df["airline_sentiment"].value_counts()

In [ ]:
sentiment_counts = df["airline_sentiment"].value_counts()

plt.figure(figsize=(8, 4))
sentiment_counts.plot(kind="bar")
plt.title("Distribución de clases")
plt.xlabel("Sentimiento")
plt.ylabel("Número de tweets")
plt.xticks(rotation=0)
plt.show()

### Lectura rápida de negocio

Ya vemos algo importante: el dataset está **desbalanceado**.  
Hay bastantes más tweets negativos que positivos.

Esto no es raro en atención al cliente. La gente suele entrar a redes a decir:

- que algo no funciona,
- que un vuelo se retrasó,
- que perdió una conexión,
- que el servicio fue malo.

Moraleja: si el modelo va "muy bien", cuidado. A veces solo está aprendiendo a predecir la clase mayoritaria.

## 6. Exploración de ejemplos reales

Vamos a leer algunos tweets reales para aterrizar el problema.

In [ ]:
pd.set_option("display.max_colwidth", 300)

df.sample(10, random_state=42)[["airline_sentiment", "airline", "text"]]

In [ ]:
for sentiment in ["negative", "neutral", "positive"]:
    print("=" * 90)
    print(f"EJEMPLOS DE LA CLASE: {sentiment.upper()}")
    print("=" * 90)
    examples = df[df["airline_sentiment"] == sentiment]["text"].sample(3, random_state=42)
    for i, txt in enumerate(examples, 1):
        print(f"{i}. {txt}")
    print()

## 7. Longitud de los textos

Antes de limpiar y vectorizar, suele ser útil ver cómo de largos son los textos.

In [ ]:
df["text_len"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()

df[["text_len", "word_count"]].describe().T

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["word_count"], bins=40)
plt.title("Distribución del número de palabras por tweet")
plt.xlabel("Número de palabras")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
df.groupby("airline_sentiment")[["text_len", "word_count"]].mean().round(2)

Esto nos da contexto: estamos ante textos **cortos**, algo muy típico en soporte, reviews, chats o redes sociales.

Eso hace que técnicas simples como **TF-IDF + modelo lineal** funcionen sorprendentemente bien muchas veces.

## 8. Preparación del texto

En NLP clásico, una parte clave del trabajo es la **limpieza** del texto.

No existe una limpieza universal perfecta. Depende del caso de negocio.

Aquí haremos una versión razonable para este problema:

- pasar a minúsculas,
- eliminar URLs,
- eliminar menciones (`@usuario`),
- quitar símbolos extraños,
- normalizar espacios.

No eliminaremos stopwords en esta primera versión, porque a veces en sentimiento ciertas palabras funcionales ayudan más de lo que parecen.

In [ ]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)     # URLs
    text = re.sub(r"@\w+", " ", text)               # menciones
    text = re.sub(r"#", " ", text)                   # quitamos solo el símbolo #
    text = re.sub(r"[^a-z\s]", " ", text)           # dejamos letras y espacios
    text = re.sub(r"\s+", " ", text).strip()        # espacios extra
    return text

df["clean_text"] = df["text"].apply(clean_text)

df[["text", "clean_text"]].sample(8, random_state=42)

### Comparativa rápida antes / después

In [ ]:
for original, cleaned in df[["text", "clean_text"]].sample(5, random_state=7).values:
    print("ORIGINAL :", original)
    print("LIMPIO   :", cleaned)
    print("-" * 120)

## 9. Definición del problema de ML

Estamos ante un problema de **clasificación multiclase**:

**Input**: el texto del tweet  
**Output**: una etiqueta entre:

- `negative`
- `neutral`
- `positive`

### Variable objetivo

In [ ]:
X = df["clean_text"]
y = df["airline_sentiment"]

print("Número de muestras:", len(X))
print("Clases:", sorted(y.unique()))

## 10. Train / test split

Separamos train y test para poder evaluar el modelo sobre datos no vistos.

Usamos `stratify=y` para mantener proporciones similares de clases en ambos conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape[0])
print("Test :", X_test.shape[0])

print("\nDistribución en train:")
print(y_train.value_counts(normalize=True).round(3))

print("\nDistribución en test:")
print(y_test.value_counts(normalize=True).round(3))

## 11. Baseline tonto pero obligatorio

Antes de ponernos finos con TF-IDF y modelos más serios, montamos un **baseline** con `DummyClassifier`.

Esto es higiene básica de ML.

Si nuestro modelo "bueno" no mejora a un clasificador tonto, tenemos un problema serio.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)

print("Accuracy baseline:", round(accuracy_score(y_test, dummy_pred), 4))
print("F1 macro baseline:", round(f1_score(y_test, dummy_pred, average="macro"), 4))
print()
print(classification_report(y_test, dummy_pred))

### Lectura rápida

El baseline probablemente acertará bastante en `accuracy` porque la clase mayoritaria pesa mucho.

Pero mira el **F1 macro**: ahí vemos mejor si el modelo está tratando bien a **todas** las clases.

En problemas desbalanceados, fiarse solo de `accuracy` es una trampa de manual.

## 12. Primer pipeline serio: TF-IDF + Logistic Regression

Este es uno de los clásicos del NLP tabularizado. Muy duro, muy fiable y nada glamuroso. Como un Toyota Corolla del texto.

### Recordatorio conceptual

`TfidfVectorizer` convierte texto en una matriz numérica donde cada columna representa un término, ponderado por su relevancia relativa.

Después, `LogisticRegression` aprende fronteras para clasificar los textos.

In [ ]:
logreg_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=8000,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95
    )),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

logreg_pipeline.fit(X_train, y_train)
logreg_pred = logreg_pipeline.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, logreg_pred), 4))
print("F1 macro:", round(f1_score(y_test, logreg_pred, average="macro"), 4))
print()
print(classification_report(y_test, logreg_pred))

## 13. Matriz de confusión del modelo LogReg

In [ ]:
cm = confusion_matrix(y_test, logreg_pred, labels=logreg_pipeline.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=logreg_pipeline.classes_)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax)
plt.title("Matriz de confusión - TF-IDF + Logistic Regression")
plt.show()

### Qué suele pasar aquí

Lo habitual es que:

- `negative` se detecte bastante bien,
- `positive` razonablemente bien,
- `neutral` sea la clase más puñetera.

Y tiene lógica: en texto real, "neutral" muchas veces se parece demasiado a uno u otro lado.

## 14. Segundo pipeline: TF-IDF + Linear SVC

Otro clásico del NLP con texto disperso (`sparse data`).

Los modelos lineales suelen funcionar muy bien en clasificación de texto cuando tenemos representaciones tipo bolsa de palabras / TF-IDF.

In [ ]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=8000,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95
    )),
    ("model", LinearSVC(class_weight="balanced"))
])

svm_pipeline.fit(X_train, y_train)
svm_pred = svm_pipeline.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, svm_pred), 4))
print("F1 macro:", round(f1_score(y_test, svm_pred, average="macro"), 4))
print()
print(classification_report(y_test, svm_pred))

## 15. Comparativa de modelos

In [ ]:
results = pd.DataFrame({
    "modelo": ["Dummy most_frequent", "TF-IDF + LogisticRegression", "TF-IDF + LinearSVC"],
    "accuracy": [
        accuracy_score(y_test, dummy_pred),
        accuracy_score(y_test, logreg_pred),
        accuracy_score(y_test, svm_pred),
    ],
    "f1_macro": [
        f1_score(y_test, dummy_pred, average="macro"),
        f1_score(y_test, logreg_pred, average="macro"),
        f1_score(y_test, svm_pred, average="macro"),
    ]
}).sort_values(by="f1_macro", ascending=False)

results.round(4)

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(results["modelo"], results["f1_macro"])
plt.title("Comparativa de modelos por F1 macro")
plt.ylabel("F1 macro")
plt.xticks(rotation=20, ha="right")
plt.show()

### Conclusión intermedia

Sin hacer nada exótico ya tenemos un pipeline bastante serio:

- dataset real,
- limpieza,
- vectorización,
- entrenamiento,
- evaluación,
- comparación de modelos.

Y esto, para muchos casos de negocio reales, ya da bastante guerra.

## 16. Interpretabilidad: palabras más influyentes por clase

Una ventaja de modelos lineales sobre TF-IDF es que podemos inspeccionar qué términos están empujando a cada clase.

Eso da **explicabilidad** y ayuda mucho al negocio.

Vamos a hacerlo con la regresión logística.

In [ ]:
tfidf = logreg_pipeline.named_steps["tfidf"]
model = logreg_pipeline.named_steps["model"]

feature_names = np.array(tfidf.get_feature_names_out())
classes = model.classes_

for i, class_name in enumerate(classes):
    top_pos = np.argsort(model.coef_[i])[-15:][::-1]
    print("=" * 100)
    print(f"PALABRAS / N-GRAMAS MÁS ASOCIADOS A LA CLASE: {class_name.upper()}")
    print("=" * 100)
    print(feature_names[top_pos])
    print()

### Lectura de negocio

Aquí suelen aparecer cosas como:

- retrasos,
- cancelaciones,
- thanks,
- great,
- delayed,
- customer service,
- hours,
- hold,
- lost baggage...

Esto ya se parece mucho a una **herramienta básica de social listening**.

## 17. Análisis de errores

Un modelo no se entiende de verdad hasta que vemos **en qué falla**.

Vamos a revisar algunos casos donde el mejor modelo se equivoca.

In [ ]:
best_model_name = results.iloc[0]["modelo"]
best_pred = svm_pred if best_model_name == "TF-IDF + LinearSVC" else logreg_pred

errors_df = pd.DataFrame({
    "text": X_test.values,
    "real": y_test.values,
    "pred": best_pred
})

errors_df = errors_df[errors_df["real"] != errors_df["pred"]].copy()
print("Número de errores:", errors_df.shape[0])

errors_df.sample(min(12, len(errors_df)), random_state=42)

### Qué puedes comentar en clase al revisar errores

Busca patrones como estos:

- ironía o sarcasmo,
- contexto insuficiente,
- texto muy corto,
- ambigüedad,
- mezcla de queja + agradecimiento,
- problemas de limpieza,
- expresiones raras o abreviadas.

Bienvenido al texto real. No era tan limpio como el dataset de iris, qué sorpresa.

## 18. Predicciones sobre mensajes nuevos

Ahora probamos el modelo con tweets / mensajes nuevos inventados por nosotros.

Esto es útil para que los alumnos vean el pipeline "en vivo".

In [ ]:
examples = [
    "thanks delta for the amazing flight and friendly staff",
    "my flight is delayed again and nobody is answering me",
    "i have a question about my baggage allowance",
    "still waiting at the airport for more information",
    "best airline experience in months"
]

best_pipeline = svm_pipeline if best_model_name == "TF-IDF + LinearSVC" else logreg_pipeline
preds = best_pipeline.predict(examples)

pd.DataFrame({
    "mensaje": examples,
    "prediccion": preds
})

## 19. Mini-caso de negocio

Imagina que trabajas en un equipo de Customer Experience y cada día entran miles de mensajes desde redes sociales.

Con un pipeline como este podrías:

- estimar sentimiento automáticamente,
- priorizar mensajes negativos,
- alimentar un dashboard diario,
- detectar picos de quejas por aerolínea,
- lanzar alertas si suben ciertos patrones.

Vamos a hacer una mini tabla operativa.

In [ ]:
demo_messages = pd.DataFrame({
    "mensaje": [
        "my bag is lost and nobody can help me",
        "thank you for solving my issue so quickly",
        "what time does boarding start for flight 208",
        "flight cancelled and terrible communication from the company",
        "great service today from the crew"
    ]
})

demo_messages["sentimiento_predicho"] = best_pipeline.predict(demo_messages["mensaje"])
demo_messages

### Posible uso en negocio

A partir de aquí podrías añadir fácilmente:

- una columna de prioridad,
- una regla de escalado,
- un dashboard en Power BI / Looker / Tableau,
- un flujo automático con email, ticketing o CRM.

Y aquí es justo donde enlaza muy bien el salto posterior a **LLMs + automatización + agentes**.

## 20. Limitaciones del enfoque clásico

Muy importante: esto funciona, sí. Pero no es magia.

### Limitaciones principales

1. **Necesitas datos etiquetados**
   - alguien tuvo que marcar tweets como positivos, negativos o neutrales.

2. **Generaliza peor fuera del dominio**
   - si entrenas con aerolíneas y luego lo pasas a banca o e-commerce, puede sufrir.

3. **Entiende poco el contexto profundo**
   - sarcasmo, ironía, dobles sentidos... mal asunto.

4. **Cambios en el lenguaje**
   - abreviaturas, modas, nuevos términos, etc.

5. **No razona**
   - clasifica por patrones estadísticos, no por "comprensión" humana.

## 21. Entonces, ¿por qué seguir enseñando NLP clásico?

Porque sigue siendo muy útil.

### Ventajas reales

- barato,
- rápido,
- interpretable,
- fácil de desplegar,
- robusto,
- perfecto para muchos casos tabulares / productivos.

Y además te da una base muy clara para entender por qué los **LLMs** han cambiado el panorama.

### Conexión con la siguiente parte de clase

Con LLMs, hoy podríamos resolver algo parecido:

- sin entrenar modelo,
- con poco o ningún dato etiquetado,
- usando prompting,
- devolviendo JSON o explicaciones,
- integrándolo en un flujo de negocio.

Pero si no entiendes este pipeline clásico, luego los LLMs parecen brujería. Y no. Son otra capa del stack.

## 22. Bonus: predicción con probabilidades (solo en LogReg)

`LinearSVC` no da probabilidades directamente de forma cómoda, pero `LogisticRegression` sí.

Esto puede ser útil para negocio cuando quieres umbrales o revisar solo casos inciertos.

In [ ]:
logreg_proba = logreg_pipeline.predict_proba(examples)

proba_df = pd.DataFrame(logreg_proba, columns=logreg_pipeline.classes_)
proba_df.insert(0, "mensaje", examples)
proba_df

### Idea práctica muy potente

Podrías decir:

- si confianza > 0.85 → auto-clasifico,
- si confianza entre 0.50 y 0.85 → revisión humana,
- si confianza < 0.50 → no automatizo.

Eso ya se parece muchísimo a un esquema **HITL (Human in the Loop)**.

## 23. Conclusiones finales

### Qué nos llevamos de este notebook

Hemos recorrido un flujo bastante completo de NLP:

- dataset público real,
- comprensión del problema,
- EDA,
- limpieza,
- representación TF-IDF,
- baseline,
- modelos lineales,
- evaluación,
- interpretabilidad,
- análisis de errores,
- conexión con negocio.

### Mensaje clave

**El NLP clásico sigue siendo útil.**  
No hace falta un LLM para todo.

Pero también hemos visto sus límites, y eso prepara el terreno perfecto para la siguiente conversación:

- prompting,
- clasificación con LLM,
- extracción estructurada,
- automatización,
- HITL,
- sistemas de IA aplicados a negocio.

## 24. Ideas de extensión opcional

Si quisieras ampliar este notebook en otra sesión, podrías probar:

1. eliminar o mantener stopwords y comparar,
2. stemming / lemmatization,
3. tuning del `TfidfVectorizer`,
4. validación cruzada,
5. topic modeling,
6. embeddings,
7. comparación con un LLM,
8. despliegue simple en una app o dashboard.

Fin del notebook. Y sí, el texto siempre da más guerra que los números. Es su hobby.